# Harmonise UK geographies

Reads every pre-downloaded boundary file from `input/`, keeps only four columns and writes one GeoPackage per geography to `output_harmonised/`:

| column | content |
|---|---|
| `data_zone_code` | the unit's own code (or GSS code, the NI trust / NHS code where no GSS code exists) |
| `data_zone_name` | the unit's own name |
| `country` | England, Wales, Scotland or Northern Ireland, derived from the code prefix (E/W/S/N; NI HSC Trust codes such as `BHSCT` are mapped to Northern Ireland) |
| `geometry` | polygons in British National Grid (EPSG:27700) |

To run the Notebook, download all required datasets in one of the following formats: GPKG, SHP, GeoJSON (.zip version is acceptable).

Parent codes, centroids, populations, Welsh names, GlobalIDs etc. are dropped, but these can be easily joined from the external data sources by the unique `data_zone_code`.

Harmonising logic follows previous developments in separate scripts and uses the following datasets openly available online:

| Dataset Name  | Source URL(s) |
|---------------|---------------------|
| 1. Small data zones (LSOA-equivalent)          |  [England and Wales](https://geoportal.statistics.gov.uk/datasets/68515293204e43ca8ab56fa13ae8a547_0/explore), [Scotland](https://maps.gov.scot/ATOM/shapefiles/SG_DataZoneBdry_2022.zip), [Northern Ireland](https://www.nisra.gov.uk/publications/small-area-boundaries-gis-format)         |
| 2. Midlle data zones (MSOA-equivalent)          |  [England and Wales](https://geoportal.statistics.gov.uk/datasets/middle-layer-super-output-areas-december-2021-boundaries-ew-bgc-v3-2), [Scotland](https://maps.gov.scot/ATOM/shapefiles/SG_IntermediateZoneBdry_2022.zip) (select `SG_IntermediateZoneBdry_2022_MHW.shp`), [Nothern Ireland](https://www.nisra.gov.uk/files/nisra/publications/geography-sdz2021-esri-shapefile.zip)           | 
| 3. Local authorities (LAD)         |   [UK](https://hub.arcgis.com/datasets/ons::local-authority-districts-december-2021-boundaries-uk-bgc/about)         |
| 4. Healthcare boards          |   [England](https://geoportal.statistics.gov.uk/datasets/5697032a83014693aec372341f9783f1_0/explore), [Wales](https://datamap.gov.wales/layers/geonode%3Alhb_high_water_mark_post_apr_2019), [Scotland](https://maps.gov.scot/ATOM/shapefiles/SG_NHS_HealthBoards_2019.zip)*, [Northern Ireland](https://admin.opendatani.gov.uk/dataset/department-of-health-trust-boundaries/resource/7fa52dde-90b8-446e-bb79-4871d1028cb4?inner_span=True)          |
| 5. Electoral wards          |    [UK](https://open-geography-portalx-ons.hub.arcgis.com/datasets/ons::wards-december-2025-boundaries-uk-bfc-1)         |
| 6. Parliamentary constituences          | [Westminster (UK)](https://geoportal.statistics.gov.uk/datasets/ons::westminster-parliamentary-constituencies-july-2024-boundaries-uk-bfc-2/about), [Senedd (Wales)](https://www.data.gov.uk/dataset/7ce6ee35-c581-4bde-b139-e8b348a3002a/senedd-cymru-constituencies-may-2026-boundaries-wa-bfc), [Holyrood (Scotland)](https://open-geography-portalx-ons.hub.arcgis.com/api/download/v1/items/68b520e3b40a4b06bcc183944de541c7/geoPackage?layers=0)          |

*Healthcare boards for Scotland are provided in at least four places, but they may return EoR (Extend of Realm) instead of MHW (Mean High Water) which is not consistent with the rest of the datasets:*
*- [Zipped shapefile, EoR type (used here)](https://maps.gov.scot/ATOM/shapefiles/SG_NHS_HealthBoards_2019.zip)*
- *[Gov Scot API, EoR type](https://maps.gov.scot/server/rest/services/ScotGov/HealthSocialCare/MapServer/0/query)*
- *[NHS Healthboards API, currently down](https://maps.data.gov.scot/arcgis/rest/services/NHSHealthBoards/MapServer/0/query)*
- *[Stirling Council, MHW type but is not masked by internal water bodies](https://data-stirling-council.hub.arcgis.com/datasets/stirling-council::open-data-nhs-health-boards-scotland/about)*

Merging logic follows previous developments in separate scripts in `geographies` [repo](https://github.com/Imago-SDRUK/geographies/tree/main/src).

**To add or change a source:** edit the config cell only (`GEOGRAPHIES`; `OUTPUT_NAMES` for the output file and layer names; `EXPECTED_COUNTS` for the count checks). Each source is a file pattern plus a regex for the code column and one for the name column (case-insensitive, so names such as `LSOA21CD`/`LSOA25CD` both match).

In [ ]:
from pathlib import Path
import re
import fnmatch
import geopandas as gpd
import pandas as pd

INPUT_DIR = Path('input')
OUTPUT_DIR = Path('geographies_harmonised')
OUTPUT_DIR.mkdir(exist_ok=True)

TARGET_CRS = 27700  # British National Grid

## Helper functions

These are functions to search for the relevant files, columns, and read them.

In [ ]:
READABLE = {'.shp', '.gpkg', '.geojson', '.json', '.zip'}


def find_file(patterns):
    """Find one boundary file under INPUT_DIR. `patterns` is a glob string or a list of glob strings"""
    patterns = [patterns] if isinstance(patterns, str) else patterns
    for pattern in patterns:
        hits = sorted(p for p in INPUT_DIR.glob(pattern)
                      if p.is_file() and p.suffix.lower() in READABLE)
        if hits:
            if len(hits) > 1:
                print(f'  note: {len(hits)} files match {pattern!r}, using {hits[0].name}')
            return hits[0]
    raise FileNotFoundError(f'Nothing in {INPUT_DIR} matches {patterns}')


def find_column(gdf, pattern):
    """Return the single column whose name fully matches the regex (case-insensitive)."""
    hits = [c for c in gdf.columns if re.fullmatch(pattern, c, flags=re.IGNORECASE)]
    if len(hits) != 1:
        raise KeyError(f'Pattern {pattern!r} matched {hits} in columns {list(gdf.columns)}')
    return hits[0]

COUNTRIES = {'E': 'England', 'W': 'Wales', 'S': 'Scotland', 'N': 'Northern Ireland'}


def country_from_code(code):
    """Country from the GSS code prefix (E/W/S/N). NI HSC Trust codes (BHSCT, NHSCT, SHSCT, SEHSCT, WHSCT)
    have no GSS prefix and some start with S/W/N, so they are matched first."""
    if code.endswith('HSCT'):
        return 'Northern Ireland'
    return COUNTRIES.get(code[:1].upper())

# so for each source we need to specify the column names (or regexes)
def harmonise(gdf, code, name, crs=None):
    """Keep only data_zone_code, data_zone_name, country, geometry; reproject to EPSG:27700."""
    if gdf.crs is None:
        gdf = gdf.set_crs(crs or TARGET_CRS)
    codes = gdf[find_column(gdf, code)].astype(str).str.strip()
    out = gpd.GeoDataFrame(
        {
            'data_zone_code': codes,
            'data_zone_name': gdf[find_column(gdf, name)].astype(str).str.strip(),
            'country': codes.map(country_from_code),
        },
        geometry=gdf.geometry,
        crs=gdf.crs,
    )
    return out.to_crs(TARGET_CRS)


def read_source(src):
    path = find_file(src['file'])
    print(f'  reading {path.name}')
    return harmonise(gpd.read_file(path), src['code'], src['name'], src.get('assumed_crs'))

## Configuration (what to read and which columns hold zone code/name)

File patterns are globs relative to `input/`, so they can be adjusted. EPSG:27700 is assumed by default, except the NISRA files (`DZ2021`, `SDZ2021`) and `dohTrustBoundary`, which are in Irish Grid (EPSG:29902).

 - First, we need to define rules to find the relevant datasets in the input folder (**GEOGRAPHIES**)
 - Second, names for the output filename (in GPKG format) and layer name should be mapped to each dataset
 - Finally, we define reference (ground truth) for the features count in each dataset

In [ ]:
GEOGRAPHIES = {

    # 1. Small areas (LSOA-equvalent)
    'uk_lsoa': [
        dict(file='**/Lower_layer_Super_Output_Areas*', code=r'LSOA21CD',   name=r'LSOA21NM'),# England & Wales
        dict(file='**/SG_DataZone_Bdry_2022*',          code=r'dzcode',     name=r'dzname'), # Scotland Data Zones
        dict(file='**/DZ2021*',                         code=r'DZ2021_cd',  name=r'DZ2021_nm', crs=29902), # NI Data Zones
    ],
    
    # 2. Middle areas (MSOA-equivalent)
    'uk_msoa': [
        dict(file='**/Middle_layer_Super_Output_Areas*',   code=r'MSOA21CD',    name=r'MSOA21NM'), # England & Wales
        dict(file='**/SG_IntermediateZoneBdry_2022_MHW*',  code=r'IZCode',      name=r'IZName'), # Scotland Intermediate Zones
        dict(file='**/SDZ2021*',                           code=r'SDZ2021_cd',  name=r'SDZ2021_nm', crs=29902),  # NI Super Data Zones
    ],

    # 3. Local auithorities (LAD-equivalent)
    'uk_lad': [
        dict(file='**/Local_Authority_Districts*', code=r'LAD\d{2}CD', name=r'LAD\d{2}NM'),
    ],

    # 4. Health boards (different body types per nation; one GeoPackage, one layer per nation - see OUTPUT_NAMES)
    'england_health_boards': [
        dict(file='**/*Sub_Integrated_Care_Board*', code=r'SICBL\d{2}CD', name=r'SICBL\d{2}NM'), # England: Sub-ICB locations
    ],
    'wales_health_boards': [
        dict(file='**/*lhb*',                       code=r'area_code',    name=r'name_en'), # Wales: Local Health Boards
    ],
    'scotland_health_boards': [
        dict(file='**/*SG_NHS_HealthBoards*',       code=r'hbcode',       name=r'hbname'), # Scotland: NHS Health Boards
    ],
    'ni_health_boards': [
        dict(file='**/*dohTrustBoundary*',          code=r'TrustCode',    name=r'TrustName', crs=29902), # NI: HSC Trusts
    ],

    # 5. Electoral wards
    'uk_wards': [
        dict(file='**/*[Ww][Dd]*2025*', code=r'WD\d{2}CD', name=r'WD\d{2}NM'),
    ],

    # 6. Constituencies
    'uk_westminster_constituencies': [
        dict(file='**/Westminster_Parliamentary_Constituencies*', code=r'PCON\d{2}CD', name=r'PCON\d{2}NM'),
    ],
    'wales_senedd_constituencies': [
        dict(file=['**/*SENC_MAY_2026*', '**/*[Ss]enedd*'],       code=r'SENC\d{2}CD', name=r'SENC\d{2}NM'),
    ],
    'scotland_holyrood_constituencies': [
        dict(file=['**/*SPC_MAY_2026*', '**/*[Hh]olyrood*'],      code=r'SPC\d{2}CD', name=r'SPC\d{2}NM'),
    ],
}


# Output GeoPackage file name (without .gpkg) and layer name inside it, per geography.
# Several geographies can share one file name: they are written as separate layers of that file.
OUTPUT_NAMES = {
    'uk_lsoa':                          ('uk_datazones', 'lsoa_2021'),
    'uk_msoa':                          ('uk_middle_datazones', 'msoa_dec2021'),
    'uk_lad':                           ('uk_loc_authorities', 'lad_dec2021'),
    'england_health_boards':            ('uk_health_boards', 'sub_icb_apr2026_EN'),
    'wales_health_boards':              ('uk_health_boards', 'lhb_2019_WA'),
    'scotland_health_boards':           ('uk_health_boards', 'hb_2014_SC'),
    'ni_health_boards':       ('uk_health_boards', 'hsc_trusts_2018_NI'),
    'uk_wards':                         ('uk_wards', 'uk_wards_dec2025'),
    'uk_westminster_constituencies':    ('uk_constituencies', 'westminster_pcon_jul2024'),
    'wales_senedd_constituencies':      ('wa_constituencies', 'senedd_may2026'),
    'scotland_holyrood_constituencies': ('sc_constituencies', 'holyrood_may2026'),
}

# Expected number of features per geography
EXPECTED_COUNTS = {
    'uk_lsoa': 46844,
    'uk_msoa': 9448,
    'uk_lad': 374,
    'england_health_boards': 106,
    'wales_health_boards': 7,
    'scotland_health_boards': 14,
    'ni_health_boards': 5,
    'uk_wards': 8405,
    'uk_westminster_constituencies': 650,
    'wales_senedd_constituencies': 16,
    'scotland_holyrood_constituencies': 73,
}

## Read, harmonise, merge, write

In [ ]:
results = {}
fresh = set()  # output files already recreated in this run (so layers can share one file)

for layer, sources in GEOGRAPHIES.items():
    print(layer)
    parts = [read_source(src) for src in sources]
    merged = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=TARGET_CRS)

    file_name, layer_name = OUTPUT_NAMES.get(layer, (layer, layer))
    out_path = OUTPUT_DIR / f'{file_name}.gpkg'
    if out_path not in fresh:
        out_path.unlink(missing_ok=True)  # overwrite cleanly on re-runs
        fresh.add(out_path)
    merged.to_file(out_path, driver='GPKG', layer=layer_name, index=False)

    results[layer] = merged
    print(f'  -> {out_path} [{layer_name}]  ({len(merged)} features)')

## Data quality checks

Every geogpraphy dataset should have exactly the four columns (`data_zone_code`, `data_zone_name`, `country`, `geometry`), a country for every feature, the expected number of features, EPSG:27700, no missing or duplicated codes, no empty or invalid geometries. This is a basic check for the output quality to spot any obvious issues.

In [ ]:
rows = []
for layer, gdf in results.items():
    rows.append({
        'layer': layer,
        'features': len(gdf),
        'columns_ok': list(gdf.columns) == ['data_zone_code', 'data_zone_name', 'country', 'geometry'],
        'crs': gdf.crs.to_epsg(),
        'null_codes': int(gdf['data_zone_code'].isin(['', 'nan', 'None']).sum()),
        'null_country': int(gdf['country'].isna().sum()),
        'duplicate_codes': int(gdf['data_zone_code'].duplicated().sum()),
        'empty_geoms': int(gdf.geometry.is_empty.sum()),
        'invalid_geoms': int((~gdf.geometry.is_valid).sum()),
    })

pd.DataFrame(rows)

In [ ]:
# Compare feature counts with the expected ones (EXPECTED_COUNTS in the config cell)
rows = []
for layer, gdf in results.items():
    expected = EXPECTED_COUNTS[layer]
    rows.append({'layer': layer, 'expected': expected, 'actual': len(gdf), 'count_ok': len(gdf) == expected})

pd.DataFrame(rows)

This also checks which features have invalid geometries and why, eg self-intersections.

In [ ]:
bad = []

for layer, gdf in results.items():
    print(f'Checking {layer}')
    mask = ~gdf.geometry.is_valid

    if mask.any():
        sub = gdf.loc[mask, ['data_zone_code', 'data_zone_name']].copy()
        sub.insert(0, 'layer', layer)
        sub['reason'] = gdf.geometry[mask].is_valid_reason() # includes the coordinates of the issue
        sub['issue'] = sub['reason'].str.replace(r'\\[.*\\]', '', regex=True)
        bad.append(sub)

invalid = pd.concat(bad, ignore_index=True) if bad else pd.DataFrame(columns=['layer', 'issue'])
print(f'{len(invalid)} invalid feature(s)')
invalid

In [ ]:
# Quick visual check of one layer
results['lsoa_uk'].plot(figsize=(6, 8), linewidth=0.05, edgecolor='black');